In [46]:
import pandas as pd
csv_path = "/Users/hiraokatatsuru/Library/Mobile Documents/com~apple~CloudDocs/postal-operation-shift-management-system/db/init/csv/postal_datas.csv"
df_raw = pd.read_csv(csv_path)
df_raw.head()

,日付,通常郵便,書留,ゆうパケット,レターパックライト,レターパックプラス,特定記録,ゆうパック,eパケット,EMS,年賀組立,年賀配達
0,2021/10/1,63000,1783,882,615,264,588,1126,200,150,0,0
1,2021/10/2,0,1951,673,337,144,0,1534,0,158,0,0
2,2021/10/3,0,1054,898,151,65,0,1177,0,142,0,0
3,2021/10/4,102000,540,688,369,158,557,1595,400,274,0,0
4,2021/10/5,45000,721,877,443,190,451,1083,150,179,0,0


In [47]:
df = df_raw[["日付", "通常郵便"]].copy()
df.head()


,日付,通常郵便
0,2021/10/1,63000
1,2021/10/2,0
2,2021/10/3,0
3,2021/10/4,102000
4,2021/10/5,45000


In [48]:
df = df.rename(columns={
    "日付": "date",
    "通常郵便": "y"
})

In [49]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)


In [50]:
df.dtypes

date    datetime64[ns]
y                int64
dtype: object

In [51]:
df.isna().sum()


date    0
y       0
dtype: int64

In [52]:
df.head()
df.tail()


,date,y
1198,2025-01-11,0
1199,2025-01-12,0
1200,2025-01-13,70000
1201,2025-01-14,41000
1202,2025-01-15,45000


In [53]:
import numpy as np
import jpholiday

In [54]:
df_feat = df.copy()

# 基本
df_feat["weekday"] = df_feat["date"].dt.weekday        # 0=月 … 6=日
df_feat["month"]   = df_feat["date"].dt.month
df_feat["year"]    = df_feat["date"].dt.year

# 祝日
df_feat["is_holiday"] = df_feat["date"].dt.date.apply(jpholiday.is_holiday).astype(int)

# 土日
df_feat["is_saturday"] = (df_feat["weekday"] == 5).astype(int)
df_feat["is_sunday"]   = (df_feat["weekday"] == 6).astype(int)

# 祝日前（※lagではない。未来参照だが実務上OK）
df_feat["is_day_before_holiday"] = (
    df_feat["date"].shift(-1).dt.date.apply(jpholiday.is_holiday).fillna(False).astype(int)
)


In [55]:
def month_to_season(m: int) -> str:
    if m in (12, 1, 2):
        return "winter"
    elif m in (3, 4, 5):
        return "spring"
    elif m in (6, 7, 8):
        return "summer"
    else:
        return "autumn"

df_feat["season"] = df_feat["month"].apply(month_to_season)


In [56]:
# 年末（12/25〜12/31）
df_feat["is_nenmatsu"] = (
    (df_feat["month"] == 12) & (df_feat["date"].dt.day >= 25)
).astype(int)

# お盆（8/13〜8/16）
df_feat["is_obon"] = (
    (df_feat["month"] == 8) & (df_feat["date"].dt.day.between(13, 16))
).astype(int)


In [57]:
train_df = df_feat[df_feat["date"] < "2025-01-15"].copy()
weekday_mean = (
    train_df
    .groupby("weekday")["y"]
    .mean()
    .rename("weekday_mean_volume")
)

df_feat = df_feat.merge(
    weekday_mean,
    on="weekday",
    how="left"
)

In [58]:
month_mean = (
    train_df
    .groupby("month")["y"]
    .mean()
    .rename("month_mean_volume")
)

df_feat = df_feat.merge(
    month_mean,
    on="month",
    how="left"
)

In [59]:
df_feat.head()


,date,y,weekday,month,year,is_holiday,is_saturday,is_sunday,is_day_before_holiday,season,is_nenmatsu,is_obon,weekday_mean_volume,month_mean_volume
0,2021-10-01,63000,4,10,2021,0,0,0,0,autumn,0,0,50610.465116,39766.129032
1,2021-10-02,0,5,10,2021,0,1,0,0,autumn,0,0,127.906977,39766.129032
2,2021-10-03,0,6,10,2021,0,0,1,0,autumn,0,0,162.790698,39766.129032
3,2021-10-04,102000,0,10,2021,0,0,0,0,autumn,0,0,77593.023256,39766.129032
4,2021-10-05,45000,1,10,2021,0,0,0,0,autumn,0,0,48308.139535,39766.129032


In [60]:
target_col = "y"
feature_cols = [
    "weekday",
    "month",
    "year",
    "is_holiday",
    "is_saturday",
    "is_sunday",
    "is_day_before_holiday",
    "is_nenmatsu",
    "is_obon",
    "weekday_mean_volume",
    "month_mean_volume",
]

X = df_feat[feature_cols]
y = df_feat[target_col]


In [61]:
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
import pandas as pd

# split 日付
train_end = pd.Timestamp("2024-09-30")
valid_end = pd.Timestamp("2024-12-31")

is_train = df_feat["date"] <= train_end
is_valid = (df_feat["date"] > train_end) & (df_feat["date"] <= valid_end)
is_test  = df_feat["date"] > valid_end

X_train, y_train = X.loc[is_train], y.loc[is_train]
X_valid, y_valid = X.loc[is_valid], y.loc[is_valid]
X_test,  y_test  = X.loc[is_test],  y.loc[is_test]

print("train:", X_train.shape)
print("valid:", X_valid.shape)
print("test :", X_test.shape)


train: (1096, 11)
valid: (92, 11)
test : (15, 11)


In [62]:
model = XGBRegressor(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False,
)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=800, n_jobs=-1,
             num_parallel_tree=None, random_state=42, ...)

In [63]:
pred_train = model.predict(X_train)
pred_valid = model.predict(X_valid)

print("MAE train:", mean_absolute_error(y_train, pred_train))
print("MAE valid:", mean_absolute_error(y_valid, pred_valid))

if len(X_test) > 0:
    pred_test = model.predict(X_test)
    print("MAE test :", mean_absolute_error(y_test, pred_test))


MAE train: 5297.894194070984
MAE valid: 9781.838515779247
MAE test : 13668.44561106364


In [64]:
pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)


is_sunday                0.296805
weekday                  0.217357
is_saturday              0.155099
is_holiday               0.151716
weekday_mean_volume      0.101760
month_mean_volume        0.019084
is_obon                  0.015065
month                    0.014428
year                     0.013724
is_nenmatsu              0.010539
is_day_before_holiday    0.004423
dtype: float32

In [65]:
forecast_start = pd.Timestamp("2025-01-16")
n_days = 28

df_future = pd.DataFrame({
    "date": pd.date_range(forecast_start, periods=n_days, freq="D")
})

df_future

,date
0,2025-01-16
1,2025-01-17
2,2025-01-18
3,2025-01-19
4,2025-01-20
5,2025-01-21
6,2025-01-22
7,2025-01-23
8,2025-01-24
9,2025-01-25


In [66]:
import jpholiday

df_future_feat = df_future.copy()

df_future_feat["weekday"] = df_future_feat["date"].dt.weekday
df_future_feat["month"]   = df_future_feat["date"].dt.month
df_future_feat["year"]    = df_future_feat["date"].dt.year

df_future_feat["is_holiday"] = (
    df_future_feat["date"].dt.date.apply(jpholiday.is_holiday).astype(int)
)

df_future_feat["is_saturday"] = (df_future_feat["weekday"] == 5).astype(int)
df_future_feat["is_sunday"]   = (df_future_feat["weekday"] == 6).astype(int)

df_future_feat["is_day_before_holiday"] = (
    df_future_feat["date"].shift(-1).dt.date
    .apply(jpholiday.is_holiday)
    .fillna(False)
    .astype(int)
)

df_future_feat["is_nenmatsu"] = (
    (df_future_feat["month"] == 12) &
    (df_future_feat["date"].dt.day >= 25)
).astype(int)

df_future_feat["is_obon"] = (
    (df_future_feat["month"] == 8) &
    (df_future_feat["date"].dt.day.between(13, 16))
).astype(int)


In [67]:
# すでに作ってあるものを再利用
# weekday_mean_volume
# month_mean_volume

df_future_feat = df_future_feat.merge(
    weekday_mean,
    on="weekday",
    how="left"
)

df_future_feat = df_future_feat.merge(
    month_mean,
    on="month",
    how="left"
)


In [68]:
df_future_feat.isna().sum()


date                     0
weekday                  0
month                    0
year                     0
is_holiday               0
is_saturday              0
is_sunday                0
is_day_before_holiday    0
is_nenmatsu              0
is_obon                  0
weekday_mean_volume      0
month_mean_volume        0
dtype: int64

In [71]:
df_adj = df_forecast.merge(
    df_feat[["date", "is_saturday", "is_sunday", "is_holiday"]],
    on="date",
    how="left"
)

df_adj["is_non_delivery"] = (
    (df_adj["is_saturday"] == 1) |
    (df_adj["is_sunday"] == 1) |
    (df_adj["is_holiday"] == 1)
)


In [72]:
df_adj["adjusted_volume"] = df_adj["forecast_volume"]

carry = 0.0
for i in range(len(df_adj)):
    if df_adj.loc[i, "is_non_delivery"]:
        carry += df_adj.loc[i, "adjusted_volume"]
        df_adj.loc[i, "adjusted_volume"] = 0.0
    else:
        df_adj.loc[i, "adjusted_volume"] += carry
        carry = 0.0


In [73]:
df_adj["forecast_volume"] = (
    (df_adj["adjusted_volume"] / 1000).round() * 1000
).astype(int)


In [74]:
forecast_start = pd.Timestamp("2025-01-16")
n_days = 28

df_future = pd.DataFrame({
    "date": pd.date_range(forecast_start, periods=n_days, freq="D")
})

In [75]:

df_future_feat = df_future.copy()

df_future_feat["weekday"] = df_future_feat["date"].dt.weekday
df_future_feat["month"]   = df_future_feat["date"].dt.month
df_future_feat["year"]    = df_future_feat["date"].dt.year

df_future_feat["is_holiday"] = (
    df_future_feat["date"].dt.date.apply(jpholiday.is_holiday).astype(int)
)
df_future_feat["is_saturday"] = (df_future_feat["weekday"] == 5).astype(int)
df_future_feat["is_sunday"]   = (df_future_feat["weekday"] == 6).astype(int)

df_future_feat["is_day_before_holiday"] = (
    df_future_feat["date"].shift(-1).dt.date
    .apply(jpholiday.is_holiday)
    .fillna(False)
    .astype(int)
)

df_future_feat["is_nenmatsu"] = (
    (df_future_feat["month"] == 12) &
    (df_future_feat["date"].dt.day >= 25)
).astype(int)

df_future_feat["is_obon"] = (
    (df_future_feat["month"] == 8) &
    (df_future_feat["date"].dt.day.between(13, 16))
).astype(int)

# train由来の集計特徴量をmerge（再計算しない）
df_future_feat = df_future_feat.merge(weekday_mean, on="weekday", how="left")
df_future_feat = df_future_feat.merge(month_mean,   on="month",   how="left")


In [76]:
X_future = df_future_feat[feature_cols]
df_future_feat["forecast_raw"] = model.predict(X_future)


In [78]:
df_adj = df_future_feat[[
    "date", "forecast_raw",
    "is_saturday", "is_sunday", "is_holiday"
]].copy()

df_adj["is_non_delivery"] = (
    (df_adj["is_saturday"] == 1) |
    (df_adj["is_sunday"] == 1) |
    (df_adj["is_holiday"] == 1)
)

df_adj["adjusted_volume"] = df_adj["forecast_raw"].astype("float64")

carry = 0.0
for i in range(len(df_adj)):
    if df_adj.loc[i, "is_non_delivery"]:
        carry += df_adj.loc[i, "adjusted_volume"]
        df_adj.loc[i, "adjusted_volume"] = 0.0
    else:
        df_adj.loc[i, "adjusted_volume"] += carry
        carry = 0.0


In [79]:
(df_adj["adjusted_volume"] / 1000).round() * 1000


0      36000.0
1      42000.0
2          0.0
3          0.0
4      82000.0
5      33000.0
6      38000.0
7      36000.0
8      42000.0
9          0.0
10         0.0
11     82000.0
12     33000.0
13     38000.0
14     36000.0
15     42000.0
16         0.0
17         0.0
18    100000.0
19     52000.0
20     43000.0
21     49000.0
22     49000.0
23         0.0
24         0.0
25     95000.0
26         0.0
27     57000.0
Name: adjusted_volume, dtype: float64

In [80]:
df_result = df_adj[["date", "forecast_volume"]]
df_result


KeyError: "['forecast_volume'] not in index"

In [83]:
from posms.models_lagless.normal import train_and_forecast_28d_from_csv

out, metrics = train_and_forecast_28d_from_csv(
    csv_path,
    forecast_start="2025-01-16",
    days=28,
    train_end="2024-09-30",
    valid_end="2024-12-31",
)

metrics
out.head()


,date,forecast_volume
0,2025-01-16,36000
1,2025-01-17,42000
2,2025-01-18,0
3,2025-01-19,0
4,2025-01-20,82000
